# Location Fitness Score Framework: Generalization Template

How to apply the Manhattan Starbucks spatial analysis to **Chicago, Tokyo, London, or your own city**.
This notebook provides a reusable pipeline — swap in your city's data sources and run the same analysis.

---

**Prerequisites:** `pandas`, `geopandas`, `scipy`, `numpy`, `requests`, `shapely`, `folium`


## 0. City Configuration

Every city-specific parameter lives in **one dictionary**.
Change this dict and the rest of the pipeline adapts automatically.


In [ ]:
# ── Example: Chicago ─────────────────────────────────────────────────
CITY_CONFIG = {
    "city_name": "Chicago",
    "brand": "Starbucks",
    "brand_osm_tag": {"brand": "Starbucks"},
    "bbox": {"south": 41.644, "west": -87.940, "north": 42.023, "east": -87.524},
    "ref_lat": 41.88,
    "transit_source": "CTA L-Train Ridership",
    "building_source": "Cook County Assessor",
    "census_year": "2022",
    "competitor_amenity": "cafe",
    "demand_weights": {"transit": 0.50, "pedestrian": 0.25, "population": 0.25},
    "cannibalization_buffer_m": 400,
}
print(f"Target city: {CITY_CONFIG['city_name']}")

### Data-Source Equivalents by City

| Data Layer | Manhattan | Chicago | London | Tokyo |
|---|---|---|---|---|
| **Store locations** | OSM `brand=Starbucks` | OSM `brand=Starbucks` | OSM `brand=Starbucks` | OSM `brand=Starbucks` |
| **Transit ridership** | MTA Subway Turnstiles | CTA L-Train Ridership | TfL Tube Entry/Exit | MLIT Station Passengers |
| **Pedestrian counts** | NYC DOT Bi-Annual | Chicago CDOT | TfL Pedestrian Counts | — (proxy via transit) |
| **Census / population** | US Census ACS (tract) | US Census ACS (tract) | ONS Census (LSOA) | e-Stat Census (chocho) |
| **Building footprints** | NYC PLUTO / MapPLUTO | Cook County Assessor | OS MasterMap | GSI Building Data |
| **Competitors** | OSM `amenity=cafe` | OSM `amenity=cafe` | OSM `amenity=cafe` | OSM `amenity=cafe` |


## 1. Step 1 — Collect Store Locations from OSM

The [Overpass API](https://overpass-api.de/) lets us query OpenStreetMap for any brand in any bounding box.
We define two helpers: one for the target brand, one for competitors.


In [ ]:
import requests
import pandas as pd

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def _overpass_query(bbox: dict, tags: dict) -> pd.DataFrame:
    """Run an Overpass QL query and return a DataFrame of (name, lat, lon)."""
    b = bbox
    tag_filter = "".join(f'["{k}"="{v}"]' for k, v in tags.items())
    query = f"""
    [out:json][timeout:30];
    (
      node{tag_filter}({b['south']},{b['west']},{b['north']},{b['east']});
      way{tag_filter}({b['south']},{b['west']},{b['north']},{b['east']});
    );
    out center;
    """
    resp = requests.get(OVERPASS_URL, params={"data": query}, timeout=60)
    resp.raise_for_status()
    elements = resp.json().get("elements", [])
    rows = []
    for e in elements:
        lat = e.get("lat") or e.get("center", {}).get("lat")
        lon = e.get("lon") or e.get("center", {}).get("lon")
        rows.append({"name": e.get("tags", {}).get("name", ""), "lat": lat, "lon": lon})
    return pd.DataFrame(rows)


def query_osm_stores(config: dict) -> pd.DataFrame:
    """Fetch target-brand stores inside the configured bounding box."""
    return _overpass_query(config["bbox"], config["brand_osm_tag"])


def query_osm_competitors(config: dict) -> pd.DataFrame:
    """Fetch competitor cafés (amenity=cafe) inside the bounding box."""
    return _overpass_query(config["bbox"], {"amenity": config["competitor_amenity"]})


# ── Usage (uncomment to run) ─────────────────────────────────────────
# stores = query_osm_stores(CITY_CONFIG)
# competitors = query_osm_competitors(CITY_CONFIG)
# print(f"{len(stores)} {CITY_CONFIG['brand']} stores, {len(competitors)} competitor cafés")

## 2. Step 2 — Enrich with Transit & Competitor Data

Use a **cKDTree** spatial join to attach the nearest transit station's ridership
to each store location, and compute a local competitor density within the
cannibalization buffer.


In [ ]:
import numpy as np
from scipy.spatial import cKDTree


def _to_xy(lat: np.ndarray, lon: np.ndarray, ref_lat: float) -> np.ndarray:
    """Convert (lat, lon) to approximate (x_m, y_m) using a local projection."""
    m_per_deg_lat = 111_320.0
    m_per_deg_lon = 111_320.0 * np.cos(np.radians(ref_lat))
    x = (lon - lon.mean()) * m_per_deg_lon
    y = (lat - lat.mean()) * m_per_deg_lat
    return np.column_stack([x, y])


def spatial_join_nearest(
    stores: pd.DataFrame,
    transit: pd.DataFrame,
    config: dict,
    value_col: str = "ridership",
) -> pd.DataFrame:
    """Attach nearest-transit value to each store row."""
    ref = config["ref_lat"]
    s_xy = _to_xy(stores["lat"].values, stores["lon"].values, ref)
    t_xy = _to_xy(transit["lat"].values, transit["lon"].values, ref)
    tree = cKDTree(t_xy)
    dist, idx = tree.query(s_xy, k=1)
    stores = stores.copy()
    stores["nearest_transit_m"] = dist
    stores[f"nearest_{value_col}"] = transit[value_col].values[idx]
    return stores


def compute_competitor_density(
    stores: pd.DataFrame,
    competitors: pd.DataFrame,
    config: dict,
) -> pd.DataFrame:
    """Count competitor cafés within cannibalization_buffer_m of each store."""
    ref = config["ref_lat"]
    buf = config["cannibalization_buffer_m"]
    s_xy = _to_xy(stores["lat"].values, stores["lon"].values, ref)
    c_xy = _to_xy(competitors["lat"].values, competitors["lon"].values, ref)
    tree = cKDTree(c_xy)
    counts = tree.query_ball_point(s_xy, r=buf)
    stores = stores.copy()
    stores["competitor_count"] = [len(c) for c in counts]
    return stores


# ── Usage (uncomment to run) ─────────────────────────────────────────
# transit = pd.read_csv("chicago_cta_ridership.csv")  # your local file
# stores = spatial_join_nearest(stores, transit, CITY_CONFIG)
# stores = compute_competitor_density(stores, competitors, CITY_CONFIG)

## 3. Step 3 — Enrich with Building & Census Data

Join census-tract population via **GeoPandas `sjoin`** and attach
building-level features (floor area, year built) via KDTree.


In [ ]:
import geopandas as gpd
from shapely.geometry import Point


def join_census(stores: pd.DataFrame, census_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Spatial-join stores to census tracts to pick up population & income."""
    gdf = gpd.GeoDataFrame(
        stores,
        geometry=[Point(xy) for xy in zip(stores["lon"], stores["lat"])],
        crs="EPSG:4326",
    )
    # Ensure CRS match
    census_gdf = census_gdf.to_crs("EPSG:4326")
    joined = gpd.sjoin(gdf, census_gdf, how="left", predicate="within")
    return joined


def join_buildings(
    stores: pd.DataFrame,
    buildings: pd.DataFrame,
    config: dict,
    value_col: str = "total_floor_area",
) -> pd.DataFrame:
    """Attach nearest building's feature to each store via KDTree."""
    ref = config["ref_lat"]
    s_xy = _to_xy(stores["lat"].values, stores["lon"].values, ref)
    b_xy = _to_xy(buildings["lat"].values, buildings["lon"].values, ref)
    tree = cKDTree(b_xy)
    _, idx = tree.query(s_xy, k=1)
    stores = stores.copy()
    stores[f"nearest_{value_col}"] = buildings[value_col].values[idx]
    return stores


# ── Usage (uncomment to run) ─────────────────────────────────────────
# census = gpd.read_file("chicago_census_tracts.geojson")
# stores_gdf = join_census(stores, census)
# buildings = pd.read_csv("cook_county_buildings.csv")
# stores = join_buildings(stores, buildings, CITY_CONFIG)

## 4. Step 4 — Compute Location Fitness Score

$$
\text{LFS} = \frac{\text{Demand Proxy Index (DPI)}}{\text{Supply Index (SI)}}
$$

where

$$
\text{DPI} = w_{\text{transit}} \cdot \hat{R}_{\text{transit}}
            + w_{\text{ped}} \cdot \hat{P}_{\text{pedestrian}}
            + w_{\text{pop}} \cdot \hat{C}_{\text{population}}
$$

$$
\text{SI} = 1 + \text{competitor\_count}
$$

All demand features are min-max normalised before weighting.


In [ ]:
def _norm(s: pd.Series) -> pd.Series:
    """Min-max normalise a Series to [0, 1]."""
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0.0


def compute_lfs(
    df: pd.DataFrame,
    demand_cols: list[str],
    weights: list[float],
    competitor_col: str = "competitor_count",
) -> pd.DataFrame:
    """
    Compute Location Fitness Score.

    Parameters
    ----------
    df : DataFrame with demand columns and competitor_col.
    demand_cols : column names for each demand signal.
    weights : corresponding weights (must sum to 1).
    competitor_col : column with count of nearby competitors.

    Returns
    -------
    DataFrame with added DPI, SI, LFS columns.
    """
    df = df.copy()
    dpi = sum(w * _norm(df[c]) for w, c in zip(weights, demand_cols))
    df["DPI"] = dpi
    df["SI"] = 1 + df[competitor_col]
    df["LFS"] = df["DPI"] / df["SI"]
    return df

In [ ]:
# ── Example usage with CITY_CONFIG ────────────────────────────────────
# demand_cols = ["nearest_ridership", "pedestrian_count", "population"]
# weights = list(CITY_CONFIG["demand_weights"].values())
# stores = compute_lfs(stores, demand_cols, weights)
# stores.sort_values("LFS", ascending=False).head(10)

## 5. Step 5 — Strategic Analysis

Three derived scores built on top of LFS:

| Score | Meaning |
|---|---|
| **Expansion Score** | High demand, low supply — promising new locations |
| **Closure Risk** | Low demand, high supply — underperforming stores |
| **Competitive White Space** | High demand, few competitors nearby |


In [ ]:
def expansion_score(df: pd.DataFrame, w_dpi: float = 0.6, w_inv_si: float = 0.4) -> pd.Series:
    """Higher when demand is strong and supply is thin."""
    return w_dpi * _norm(df["DPI"]) + w_inv_si * _norm(1 / df["SI"])


def closure_risk(df: pd.DataFrame, w_low_dpi: float = 0.5, w_si: float = 0.5) -> pd.Series:
    """Higher when demand is weak and competition is fierce."""
    return w_low_dpi * _norm(1 - _norm(df["DPI"])) + w_si * _norm(df["SI"])


def competitive_white_space(
    df: pd.DataFrame,
    w_dpi: float = 0.5,
    w_low_comp: float = 0.5,
) -> pd.Series:
    """Higher when demand is strong and few competitors exist."""
    return w_dpi * _norm(df["DPI"]) + w_low_comp * _norm(1 / df["SI"])


# ── Usage (uncomment to run) ─────────────────────────────────────────
# stores["expansion"] = expansion_score(stores)
# stores["closure_risk"] = closure_risk(stores)
# stores["white_space"] = competitive_white_space(stores)

## 6. Step 6 — Optimal Placement (Greedy Maximal Coverage)

Given a set of candidate locations and a coverage radius, greedily select
**K** new sites that maximise demand coverage while respecting a minimum
spacing constraint (cannibalization buffer).

Complexity: *O(K * N)* per iteration.


In [ ]:
def greedy_placement(
    candidates: pd.DataFrame,
    existing: pd.DataFrame,
    config: dict,
    k: int = 5,
    coverage_m: float = 500.0,
    demand_col: str = "DPI",
) -> pd.DataFrame:
    """
    Select k new store locations via greedy maximal-coverage.

    Parameters
    ----------
    candidates : DataFrame of potential sites with lat, lon, and demand_col.
    existing : DataFrame of current stores with lat, lon.
    config : CITY_CONFIG dict (needs ref_lat, cannibalization_buffer_m).
    k : number of new sites to place.
    coverage_m : radius (metres) that a new store "covers".
    demand_col : column in candidates representing demand value.

    Returns
    -------
    DataFrame of selected sites in placement order.
    """
    ref = config["ref_lat"]
    buf = config["cannibalization_buffer_m"]

    c_xy = _to_xy(candidates["lat"].values, candidates["lon"].values, ref)
    e_xy = _to_xy(existing["lat"].values, existing["lon"].values, ref)

    # Track which candidates are still eligible
    eligible = np.ones(len(candidates), dtype=bool)

    # Mark candidates too close to existing stores
    if len(e_xy) > 0:
        tree_e = cKDTree(e_xy)
        too_close = tree_e.query_ball_point(c_xy, r=buf)
        for i, hits in enumerate(too_close):
            if len(hits) > 0:
                eligible[i] = False

    selected_indices = []
    demand = candidates[demand_col].values.copy()

    for _ in range(k):
        scores = np.where(eligible, demand, -np.inf)
        best = int(np.argmax(scores))
        if scores[best] == -np.inf:
            break
        selected_indices.append(best)
        eligible[best] = False

        # Remove candidates within buffer of the newly placed store
        dists = np.linalg.norm(c_xy - c_xy[best], axis=1)
        eligible[dists < buf] = False

    return candidates.iloc[selected_indices].reset_index(drop=True)


# ── Usage (uncomment to run) ─────────────────────────────────────────
# new_sites = greedy_placement(candidate_grid, stores, CITY_CONFIG, k=5)
# print(new_sites[["lat", "lon", "DPI"]])

## 7. Adaptation Checklist

### Minimum Viable vs. Ideal Data by Step

| Step | Minimum Viable | Ideal |
|---|---|---|
| 1. Store locations | OSM Overpass query | Official franchise list + OSM |
| 2. Transit | Station coordinates only | Station-level ridership counts |
| 3. Census / population | Country-level census tracts | Block-group or LSOA granularity |
| 3. Buildings | OSM building footprints | Assessor data with floor area & year |
| 4. LFS | Two demand signals | Three or more demand signals |
| 6. Candidates | Regular grid within bbox | Parcels or vacant retail lots |

### Common Pitfalls

1. **CRS / Projection** — Always project to a local metre-based CRS (or use the `_to_xy` helper with `ref_lat`) before computing distances. WGS-84 degrees will produce wrong results.
2. **OSM Coverage** — OSM completeness varies by country. Verify brand store counts against official reports.
3. **Transit Data Format** — Ridership files differ wildly (daily vs. annual, station vs. entrance). Normalise to **average weekday entries** before feeding into the pipeline.
4. **Census Geography** — Tract / LSOA / chocho boundaries do not nest neatly across countries. Ensure your population column is density (per km²) or total with area, not raw counts of mismatched polygons.
5. **Pedestrian Data Sparsity** — Many cities lack official pedestrian counts. Proxies: transit ridership, POI density, or foot-traffic estimates from mobility data vendors.
6. **Competitor Definition Varies by Market** — In Tokyo, convenience stores (konbini) compete with Starbucks more than independent cafés. Adjust `competitor_amenity` or add multiple tags.


## Series Navigation

| # | Notebook | Link |
|---|----------|------|
| 0 | Manhattan Cafe Wars — EDA | [Kaggle](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) |
| 1 | Starbucks 10-K NLP: Topic & Keyword Analysis | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) |
| 1F | Starbucks 10-K LDA Topic Explorer (pyLDAvis) | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis) |
| 2A | Starbucks Spatial Clustering | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) |
| 2B | Starbucks Location Fitness | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) |
| 2C | Starbucks Walk-Distance Analysis (OSMnx) | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx) |
| 5A | LFS Predictive Model | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness-score-predictive-model) |
| 5B | Strategic Location Insights | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-strategic-location-insights) |
| 5C | Optimal Store Placement | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-optimal-store-placement) |
| 5D | Hourly Demand Patterns | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-hourly-demand-patterns) |
| 5E | Interactive Strategy Map | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-interactive-strategy-map) |
| **T** | **Generalization Template (this notebook)** | — |
| C | Data Pipeline: EDGAR + OSM → CSV | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) |
| D | US Expansion Animated Choropleth | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-us-expansion-animated-choropleth) |
| G | NLP × Spatial Integration | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-nlp-x-spatial) |

---

**Dataset:** [Kaggle – Starbucks Manhattan Analysis](https://www.kaggle.com/datasets/shiratoriseto/starbucks-manhattan-analysis)
**GitHub:** [github.com/seto-git/starbucks-kaggle](https://github.com/seto-git/starbucks-kaggle)
